# 03 NLP Analysis

This notebook applies the natural language processing pipeline to the stored
data. Sentiment is scored with VADER on a near-raw version of the text, a
churn-risk proxy is built from star ratings and sentiment for the Trustpilot
reviews, and the text is then processed into a separate version for keyword
extraction with TF-IDF and topic modelling with LDA.

The data is read back from the SQLite database written in the previous notebook.
Both sources are used for the descriptive, sentiment and topic analysis, while
the churn-risk proxy and the supervised stage use the Trustpilot reviews only,
since those carry a star rating.

In [58]:
import sqlite3
import pandas as pd

# Open the database written in notebook 02.
connection = sqlite3.connect('telecom_reviews.db')

# Read the whole reviews table back into a DataFrame.
df = pd.read_sql('SELECT * FROM reviews', connection)

connection.close()

# SQLite has no native date type, so the date column came back as text. Convert
# it back to a real datetime so any time-based work behaves correctly.
df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(f"Rows read from database: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
print("Rows per source:")
print(df['source'].value_counts())
print()
print("Date column type after conversion:", df['date'].dtype)

Rows read from database: 1929
Columns: ['provider', 'source', 'rating', 'title', 'text', 'date', 'url', 'text_clean', 'word_count']

Rows per source:
source
Trustpilot    1502
Geekzone       427
Name: count, dtype: int64

Date column type after conversion: datetime64[ns]


## 1. Sentiment Analysis with VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon and
rule-based sentiment analysis tool, developed and validated for short, informal
text such as social media posts and reviews (Hutto and Gilbert, 2014). It scores
text using a dictionary of pre-rated words, adjusted by rules that account for
punctuation, capitalisation, intensifiers and negation, which makes it suited to
the Trustpilot and Geekzone text in this study.

VADER is applied to the `text_clean` column, which has had structural noise
removed (URLs, glued sentence spacing) while preserving capitalisation,
punctuation and negation, the features VADER relies on to judge intensity and
polarity.

Each text receives four scores: the proportion of the text that is negative,
neutral and positive (`neg`, `neu`, `pos`), and a single compound score ranging
from -1 to +1 that summarises overall sentiment. The compound score is used in
the churn-risk proxy constructed later in this notebook.

In [59]:
# Install the VADER sentiment package. --break-system-packages is required in
# this environment's Python setup.
!pip install vaderSentiment --break-system-packages --quiet

In [60]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Create the analyser once. It loads the word dictionary into memory, so we
# only need to create it once and reuse it for every text.
analyzer = SentimentIntensityAnalyzer()

# Test on a few short, clear examples first, to see what the four scores look
# like before running this on the full dataset.
test_texts = [
    "This is the WORST service I have ever had!!! Absolutely terrible.",
    "Great experience, very happy with the support team.",
    "The technician was not helpful and the connection is still not working.",
    "It's an internet provider. It works most of the time.",
]

for text in test_texts:
    scores = analyzer.polarity_scores(text)
    print(f"Text: {text}")
    print(f"  neg={scores['neg']:.3f}  neu={scores['neu']:.3f}  "
          f"pos={scores['pos']:.3f}  compound={scores['compound']:.3f}")
    print()

Text: This is the WORST service I have ever had!!! Absolutely terrible.
  neg=0.503  neu=0.497  pos=0.000  compound=-0.878

Text: Great experience, very happy with the support team.
  neg=0.000  neu=0.317  pos=0.683  compound=0.895

Text: The technician was not helpful and the connection is still not working.
  neg=0.175  neu=0.825  pos=0.000  compound=-0.325

Text: It's an internet provider. It works most of the time.
  neg=0.000  neu=1.000  pos=0.000  compound=0.000



In [61]:
# Apply VADER to every row, using the structurally cleaned text.
# We extract all four scores into their own columns, because the proxy uses the
# compound score and the others are useful for inspection and reporting.

# polarity_scores returns a dictionary per text. We build a list of these
# dictionaries, then expand it into columns.
scores = df['text_clean'].apply(lambda t: analyzer.polarity_scores(str(t)))

# scores is now a Series of dictionaries. Turn it into a DataFrame of four
# columns and join it back to the main df.
scores_df = pd.json_normalize(scores)
scores_df.columns = ['vader_neg', 'vader_neu', 'vader_pos', 'vader_compound']

# Attach the four new columns to df, aligning by position.
df = pd.concat([df.reset_index(drop=True), scores_df], axis=1)

print("VADER applied to all rows.")
print(f"Rows scored: {len(df)}")
print()
print("New columns added:", list(scores_df.columns))
print()
print("Compound score summary:")
print(df['vader_compound'].describe())

VADER applied to all rows.
Rows scored: 1929

New columns added: ['vader_neg', 'vader_neu', 'vader_pos', 'vader_compound']

Compound score summary:
count    1929.000000
mean       -0.102977
std         0.666353
min        -0.996400
25%        -0.740400
50%        -0.244600
75%         0.557400
max         0.996200
Name: vader_compound, dtype: float64


In [62]:
# Confirm the saved Geekzone file has the new structure with dates.
import pandas as pd
check = pd.read_csv('geekzone_filtered.csv')
print(f"Rows: {len(check)}")
print(f"Columns: {list(check.columns)}")
print(f"Has date column: {'date' in check.columns}")

Rows: 427
Columns: ['provider', 'source', 'text', 'date', 'url']
Has date column: True


## 2. Churn-Risk Proxy Construction

The supervised stage requires a target variable indicating churn risk. As the
data holds no direct churn label, a proxy is constructed from the Trustpilot
star rating, which is a direct expression of customer (dis)satisfaction. The
proxy is binary and is built on the Trustpilot reviews only, since Geekzone
posts carry no rating.

Reviews of one or two stars are labelled high risk, reflecting strong
dissatisfaction. Reviews of four or five stars are labelled low risk, reflecting
satisfaction. Three-star reviews are excluded from the supervised stage, as they
express neither clear satisfaction nor clear dissatisfaction and would blur the
separation between the two classes.

The proxy is defined from the rating alone. Sentiment, although examined
alongside the rating, is not used to define the target, so that the VADER
sentiment score can serve as an independent predictor in the model without the
circularity that would arise from using sentiment both to define and to predict
the target.

In [63]:
# Work on the Trustpilot subset only, since the proxy needs a star rating.
trustpilot_df = df[df['source'] == 'Trustpilot'].copy()

# Define the binary churn-risk proxy from the rating alone.
#   1-2 stars -> high risk (1)
#   4-5 stars -> low risk (0)
#   3 stars   -> excluded (set to a missing value, then dropped)
def assign_risk(rating):
    if rating in (1, 2):
        return 1          # high risk
    elif rating in (4, 5):
        return 0          # low risk
    else:
        return None       # 3 stars, to be excluded

trustpilot_df['churn_risk'] = trustpilot_df['rating'].apply(assign_risk)

# Report before excluding 3-star reviews.
print("Churn-risk proxy assigned (before excluding 3-star reviews):")
print(f"  Total Trustpilot reviews: {len(trustpilot_df)}")
print(f"  3-star reviews to exclude: {trustpilot_df['churn_risk'].isna().sum()}")
print()

# Build the classification dataset: drop the excluded 3-star rows.
classification_df = trustpilot_df[trustpilot_df['churn_risk'].notna()].copy()
classification_df['churn_risk'] = classification_df['churn_risk'].astype(int)

print(f"Classification dataset size: {len(classification_df)}")
print()
print("Class distribution:")
print(classification_df['churn_risk'].value_counts())
print()
print("Class distribution as proportion:")
print(classification_df['churn_risk'].value_counts(normalize=True).round(3))

Churn-risk proxy assigned (before excluding 3-star reviews):
  Total Trustpilot reviews: 1502
  3-star reviews to exclude: 19

Classification dataset size: 1483

Class distribution:
churn_risk
1    1330
0     153
Name: count, dtype: int64

Class distribution as proportion:
churn_risk
1    0.897
0    0.103
Name: proportion, dtype: float64


## 3. Text Pre-processing for TF-IDF and LDA

TF-IDF and LDA require a different version of the text from the one used by
VADER. Where VADER needs capitalisation, punctuation and negation, these methods
need the text reduced to its content words, so that terms can be counted and
grouped by meaning rather than by surface form.

A separate pre-processed version of the text is built for this purpose. The text
is lowercased, punctuation and numbers are removed, stopwords (common function
words such as "the" and "is") are removed, and each remaining word is lemmatised,
reduced to its base form, so that "billing", "billed" and "bills" are counted as
one term. Quoted text in Geekzone posts is removed so that words belonging to a
quoted post do not inflate the counts of the quoting post. Emojis are removed at
this stage, since they carry sentiment rather than topic.

The pre-processing uses spaCy, whose lemmatisation is based on each word's part
of speech, which groups word variants more accurately and produces cleaner
keywords and topics.

In [64]:
# Step 1: install the spaCy library (the engine).
!pip install spacy --break-system-packages --quiet

# Step 2: download the English language model. 
!python -m spacy download en_core_web_sm --quiet

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [65]:
import spacy

# Load the English model. 'nlp' tokenises, tags parts of speech, and lemmatises.
# The parser and named-entity recogniser are disabled, as they are not needed.
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

print("spaCy English model loaded.")
print("Pipeline components in use:", nlp.pipe_names)

spaCy English model loaded.
Pipeline components in use: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']


In [66]:
import re

def remove_geekzone_quote_prefix(text):
    """
    Remove a leading quote prefix from a Geekzone post, conservatively.
    Geekzone quotes appear at the start as 'Username:' with no spaces. This
    removes one or more such prefixes from the start only.
    """
    pattern = re.compile(r'^[A-Za-z0-9_]+:')
    while pattern.match(text):
        text = pattern.sub('', text, count=1)
    return text


def preprocess_for_topics(text, source):
    """
    Build the content-word version of a text for TF-IDF and LDA.

    Steps:
      1. For Geekzone posts, remove a leading quote prefix.
      2. Process the text with spaCy.
      3. Keep only content tokens: drop stopwords, punctuation, numbers, spaces
         and any token that is not alphabetic.
      4. Lemmatise each kept token and lowercase it.
    Returns a single string of space-separated lemmas.
    """
    text = str(text)

    # 1. Conservative quote-prefix removal, Geekzone only.
    if source == 'Geekzone':
        text = remove_geekzone_quote_prefix(text)

    # 2. Process with spaCy.
    doc = nlp(text)

    # 3 and 4. Keep content tokens, lemmatise, lowercase.
    #   token.is_stop   -> True for stopwords like 'the', 'is'
    #   token.is_punct  -> True for punctuation
    #   token.is_alpha  -> True only if the token is letters (drops numbers,
    #                      emojis, mixed tokens)
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    ]

    return ' '.join(lemmas)

In [67]:
# Test the first version of the pre-processing on a few examples.
test_cases = [
    ("The BILLING was terrible! They overcharged me twice and the support team was not helpful at all.", "Trustpilot"),
    ("linw:How come you get $75 when their site says it is $97 on a 12 mth plan?", "Geekzone"),
    ("Cancelled my contract after years of poor service. Switching to another provider.", "Trustpilot"),
]

for text, source in test_cases:
    processed = preprocess_for_topics(text, source)
    print(f"SOURCE: {source}")
    print(f"BEFORE: {text}")
    print(f"AFTER:  {processed}")
    print("-" * 70)

SOURCE: Trustpilot
BEFORE: The BILLING was terrible! They overcharged me twice and the support team was not helpful at all.
AFTER:  billing terrible overcharge twice support team helpful
----------------------------------------------------------------------
SOURCE: Geekzone
BEFORE: linw:How come you get $75 when their site says it is $97 on a 12 mth plan?
AFTER:  come site say mth plan
----------------------------------------------------------------------
SOURCE: Trustpilot
BEFORE: Cancelled my contract after years of poor service. Switching to another provider.
AFTER:  cancel contract year poor service switch provider
----------------------------------------------------------------------


In [68]:
# Apply the topic pre-processing to every row. This uses spaCy on each text, so
# it takes a little longer than the earlier steps. For 1929 rows it should
# finish in well under a minute.
df['text_topics_v1'] = df.apply(
    lambda row: preprocess_for_topics(row['text_clean'], row['source']),
    axis=1
)

# Count content words in the processed text, for the LDA minimum-length rule.
df['content_word_count'] = df['text_topics_v1'].apply(lambda t: len(t.split()))

print("Topic pre-processing applied to all rows.")
print(f"Rows processed: {len(df)}")
print()
print("Content word count summary:")
print(df['content_word_count'].describe())
print()
print("By source:")
print(df.groupby('source')['content_word_count'].describe().round(1))

Topic pre-processing applied to all rows.
Rows processed: 1929

Content word count summary:
count    1929.000000
mean       36.620010
std        36.263762
min         1.000000
25%        16.000000
50%        27.000000
75%        44.000000
max       362.000000
Name: content_word_count, dtype: float64

By source:
             count  mean   std  min   25%   50%   75%    max
source                                                      
Geekzone     427.0  37.7  39.3  1.0  13.0  25.0  48.0  271.0
Trustpilot  1502.0  36.3  35.4  1.0  16.0  28.0  44.0  362.0


### 3.1 Refining the Pre-processing for Cleaner Drivers

An initial TF-IDF pass showed two sources of noise in the ranked terms. Generic
verbs such as "go", "get" and "try" ranked highly without naming any driver, and
domain terms that appear throughout the corpus, such as "nz" and the operator
names, added no distinguishing value. The pre-processing is refined to address
both.

Terms are restricted to nouns and adjectives, which name the services, problems
and qualities that act as churn drivers, with a short list of churn-related verbs
(such as "cancel", "switch" and "leave") kept because they signal churn intent
directly. Generic verbs are dropped. A custom stopword list removes geographic
and operator names (such as "nz", "spark", "one nz", "2degrees" and "vodafone"),
which appear across the whole corpus and therefore distinguish nothing.

In [69]:
# Custom domain stopwords: geographic and operator names that appear across the
# whole corpus and so distinguish nothing. Kept lowercase to match lemmas.
domain_stopwords = {
    'nz', 'new', 'zealand', 'spark', 'one', 'vodafone', 'degree', 'degrees',
    '2degrees', 'skinny', 'company'
}

# Churn-related verbs to keep even though they are verbs, because they signal
# churn intent. These are lemma forms.
keep_verbs = {
    'cancel', 'switch', 'leave', 'overcharge', 'port', 'disconnect',
    'terminate', 'refund', 'charge'
}

def preprocess_for_topics_v2(text, source):
    """
    Refined Version
    Changes from the first version:
      - Keep only nouns and adjectives, plus a short list of churn-related verbs.
      - Remove domain stopwords (operator and place names).
    """
    text = str(text)

    if source == 'Geekzone':
        text = remove_geekzone_quote_prefix(text)

    doc = nlp(text)

    kept = []
    for token in doc:
        if not token.is_alpha or token.is_stop:
            continue
        lemma = token.lemma_.lower()
        if lemma in domain_stopwords:
            continue
        # Keep nouns and adjectives, or a churn-related verb.
        if token.pos_ in ('NOUN', 'ADJ'):
            kept.append(lemma)
        elif token.pos_ == 'VERB' and lemma in keep_verbs:
            kept.append(lemma)

    return ' '.join(kept)

In [70]:
# Reprocess all rows with the refined function.
df['text_topics'] = df.apply(
    lambda row: preprocess_for_topics_v2(row['text_clean'], row['source']),
    axis=1
)

# Recount content words after refinement.
df['content_word_count'] = df['text_topics'].apply(lambda t: len(t.split()))

# Rebuild the topic corpus with the same 10-word minimum.
topic_corpus = df[df['content_word_count'] >= MIN_CONTENT_WORDS].copy()

print("Reprocessing complete.")
print(f"Topic corpus size after refinement: {len(topic_corpus)} rows")
print()
print("Content word count summary after refinement:")
print(df['content_word_count'].describe().round(1))
print()
print("Corpus by source:")
print(topic_corpus['source'].value_counts())

Reprocessing complete.
Topic corpus size after refinement: 1379 rows

Content word count summary after refinement:
count    1929.0
mean       20.5
std        19.9
min         0.0
25%         9.0
50%        15.0
75%        26.0
max       215.0
Name: content_word_count, dtype: float64

Corpus by source:
source
Trustpilot    1103
Geekzone       276
Name: count, dtype: int64


### 3.2 Building the Topic-Modelling Corpus

The pre-processed text is filtered to build the corpus used for topic modelling,
since documents with very few content words give the topic model little to work
with. The minimum length was chosen after examining the corpus size across a
range of thresholds. The corpus shrinks at a steady rate as the threshold rises,
with no sharp cut-off point, so the choice reflects a balance between retaining
documents and giving each enough content for topic inference. A minimum of ten
content words was selected, which keeps a large corpus (1,379 documents) while
ensuring each document carries enough substance for the topic model.

This minimum is applied only to the topic-modelling corpus. Shorter reviews
remain in the dataset for sentiment analysis, the churn-risk proxy and the
classification, where their length does not reduce their value, since a short
review with a clear rating and sentiment is still a valid signal. This matters
in this domain, where brief, strongly worded reviews often carry concentrated
churn signal. Both sources are included, since the topic modelling and thematic
analysis draw on Trustpilot and Geekzone together.

In [71]:
# Measure the topic corpus size at different content-word thresholds, to choose
# the minimum with the numbers in front of us rather than in the abstract.
print("Effect of different content-word thresholds on the topic corpus:")
print()
print(f"{'Threshold':>10} | {'Corpus size':>12} | {'Trustpilot':>11} | {'Geekzone':>9} | {'% of full':>9}")
print("-" * 65)

for thr in [5, 6, 7, 8, 9, 10, 12, 15]:
    subset = df[df['content_word_count'] >= thr]
    n = len(subset)
    n_tp = len(subset[subset['source'] == 'Trustpilot'])
    n_gz = len(subset[subset['source'] == 'Geekzone'])
    pct = n / len(df) * 100
    print(f"{thr:>10} | {n:>12} | {n_tp:>11} | {n_gz:>9} | {pct:>8.1f}%")

Effect of different content-word thresholds on the topic corpus:

 Threshold |  Corpus size |  Trustpilot |  Geekzone | % of full
-----------------------------------------------------------------
         5 |         1771 |        1400 |       371 |     91.8%
         6 |         1700 |        1352 |       348 |     88.1%
         7 |         1630 |        1304 |       326 |     84.5%
         8 |         1551 |        1241 |       310 |     80.4%
         9 |         1462 |        1170 |       292 |     75.8%
        10 |         1379 |        1103 |       276 |     71.5%
        12 |         1214 |         978 |       236 |     62.9%
        15 |          995 |         802 |       193 |     51.6%


In [72]:
# Build the topic-modelling corpus: keep texts with at least 10 content words.
MIN_CONTENT_WORDS = 10

topic_corpus = df[df['content_word_count'] >= MIN_CONTENT_WORDS].copy()

print(f"Full dataset: {len(df)} rows")
print(f"Topic-modelling corpus (>= {MIN_CONTENT_WORDS} content words): {len(topic_corpus)} rows")
print(f"Excluded as too short for topic modelling: {len(df) - len(topic_corpus)} rows")
print()
print("Topic corpus by source:")
print(topic_corpus['source'].value_counts())
print()
print("Topic corpus by provider:")
print(topic_corpus['provider'].value_counts())

Full dataset: 1929 rows
Topic-modelling corpus (>= 10 content words): 1379 rows
Excluded as too short for topic modelling: 550 rows

Topic corpus by source:
source
Trustpilot    1103
Geekzone       276
Name: count, dtype: int64

Topic corpus by provider:
provider
2degrees    528
One NZ      509
Spark       342
Name: count, dtype: int64


## 4. TF-IDF Keyword Extraction (Exploratory)

TF-IDF (Term Frequency, Inverse Document Frequency) weighs each term by how
often it appears in a document against how widely it appears across the whole
collection. Terms that occur often in a document but rarely elsewhere receive
the highest weight, since they distinguish that document from the rest. Applied
across the feedback, TF-IDF produces a ranked view of the terms that most
characterise the text, which supports the identification of churn-risk drivers.

Two analyses are run. The first ranks the most distinctive terms across the
whole corpus, giving an overview of the prominent terms in the feedback. The
second compares the terms that characterise high-risk reviews against low-risk
reviews, using the Trustpilot subset where the churn-risk proxy provides the
labels, which speaks directly to the question of which factors appear most in
high-risk feedback.

Both unigrams and bigrams are included, so that single words such as "billing"
and expressions such as "customer service" are captured.

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build the TF-IDF model over the topic corpus (the version with content words).
# Parameters explained:
#   ngram_range=(1, 2) -> include unigrams (single words) and bigrams (pairs)
#   min_df=5           -> ignore terms appearing in fewer than 5 documents, to
#                         drop rare noise and misspellings
#   max_df=0.5         -> ignore terms appearing in more than 50% of documents,
#                         since terms that common do not distinguish anything
#   max_features=1000  -> keep the 1000 highest-weighted terms, to stay manageable
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.5,
    max_features=1000
)

# Fit the model on the topic corpus and transform it into a TF-IDF matrix.
tfidf_matrix = vectorizer.fit_transform(topic_corpus['text_topics'])

# To rank terms overall, sum each term's TF-IDF weight across all documents.
# A higher total means the term is both frequent and distinctive across the corpus.
import numpy as np
term_names = vectorizer.get_feature_names_out()
term_scores = np.asarray(tfidf_matrix.sum(axis=0)).ravel()

# Put terms and scores in a table, sorted by score.
tfidf_overall = pd.DataFrame({
    'term': term_names,
    'tfidf_score': term_scores
}).sort_values('tfidf_score', ascending=False).reset_index(drop=True)

print("Top 30 terms by TF-IDF across the whole corpus:")
print(tfidf_overall.head(30).to_string(index=False))

Top 30 terms by TF-IDF across the whole corpus:
            term  tfidf_score
         service    84.426728
        customer    74.512345
           phone    72.148584
            time    54.857726
            plan    52.689152
customer service    50.888754
           month    45.025847
           issue    41.441782
             bad    41.110336
         account    41.079135
             day    41.046102
          charge    37.883706
        provider    36.107425
        internet    36.035644
          number    35.041082
            year    34.610833
            hour    30.630175
      connection    29.137355
            bill    28.641732
           modem    28.159439
          mobile    28.067782
           datum    27.777237
           email    27.126402
         problem    26.999650
       broadband    26.468331
            good    26.445636
           money    26.282675
            week    26.117172
          cancel    25.847205
           store    25.672075


### 4.1 High-Risk versus Low-Risk Terms

To address which factors appear most in high-risk feedback, the terms that
characterise high-risk reviews are compared against those that characterise
low-risk reviews. This uses the Trustpilot subset, where the churn-risk proxy
provides the labels. High-risk reviews are those rated one or two stars, and
low-risk reviews those rated four or five.

For each term, the proportion of high-risk reviews and of low-risk reviews that
contain it is computed. The difference between these proportions shows which
terms are distinctive of each group. Terms far more common in high-risk reviews
are candidate churn drivers, while terms more common in low-risk reviews
characterise satisfied customers. TF-IDF weighting is then used to rank the
high-risk terms by importance.

In [74]:
# Build the labelled Trustpilot set for this comparison. We need the refined
# processed text (text_topics) together with the churn_risk label.
# The classification dataset built in section 2 has the label; we join the
# processed text from df by aligning on the same Trustpilot rows.

# Take Trustpilot rows that have a churn_risk label (1-2 or 4-5 stars).
labelled = df[df['source'] == 'Trustpilot'].copy()
labelled['churn_risk'] = labelled['rating'].apply(
    lambda r: 1 if r in (1, 2) else (0 if r in (4, 5) else None)
)
labelled = labelled[labelled['churn_risk'].notna()].copy()
labelled['churn_risk'] = labelled['churn_risk'].astype(int)

# Also require some processed content, so empty processed texts do not distort
# the proportions.
labelled = labelled[labelled['text_topics'].str.split().str.len() >= 1].copy()

high_risk = labelled[labelled['churn_risk'] == 1]
low_risk = labelled[labelled['churn_risk'] == 0]

print(f"Labelled Trustpilot reviews for this analysis: {len(labelled)}")
print(f"  High risk (1-2 stars): {len(high_risk)}")
print(f"  Low risk (4-5 stars):  {len(low_risk)}")

Labelled Trustpilot reviews for this analysis: 1481
  High risk (1-2 stars): 1328
  Low risk (4-5 stars):  153


In [75]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Use a presence/absence count: for each term, in how many reviews of each group
# it appears. binary=True counts each term once per review, so we measure the
# proportion of reviews containing the term, not raw frequency.
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    min_df=5,          # ignore very rare terms
    binary=True        # presence per review, not count
)

# Fit on all labelled reviews so both groups share the same vocabulary.
X = vectorizer.fit_transform(labelled['text_topics'])
terms = vectorizer.get_feature_names_out()

# Split the matrix into the two groups using the label.
is_high = (labelled['churn_risk'] == 1).values
is_low = (labelled['churn_risk'] == 0).values

# Proportion of reviews in each group that contain each term.
high_prop = np.asarray(X[is_high].mean(axis=0)).ravel()
low_prop = np.asarray(X[is_low].mean(axis=0)).ravel()

# Build a table with the two proportions and their difference.
comparison = pd.DataFrame({
    'term': terms,
    'high_risk_pct': (high_prop * 100).round(1),
    'low_risk_pct': (low_prop * 100).round(1),
    'difference': ((high_prop - low_prop) * 100).round(1)
})

# Terms most distinctive of HIGH risk: largest positive difference.
print("Top 25 terms more common in HIGH-RISK reviews:")
print(comparison.sort_values('difference', ascending=False).head(25).to_string(index=False))

Top 25 terms more common in HIGH-RISK reviews:
            term  high_risk_pct  low_risk_pct  difference
             bad           23.3           3.3        20.0
        customer           45.9          30.1        15.8
           month           18.8           3.9        14.9
            time           28.2          13.7        14.5
        provider           18.0           3.9        14.1
         account           16.3           2.6        13.7
          charge           16.2           3.3        12.9
            hour           14.7           3.3        11.4
customer service           31.6          20.3        11.3
          cancel           11.4           0.7        10.8
           email           12.0           1.3        10.7
             day           18.3           8.5         9.8
           money           11.5           2.0         9.6
        terrible            9.9           0.7         9.3
          number           11.3           2.6         8.7
            week         

The mirror view shows the terms more common in low-risk reviews, characterising
satisfied customers. These terms indicate what customers who rate the service
highly tend to mention, which gives contrast to the high-risk drivers and
informs the discussion of what retains customers and what smaller providers
might offer to attract them.

In [76]:
# Terms most distinctive of LOW risk: largest negative difference means the term
# is far more common in low-risk than in high-risk reviews. We sort ascending on
# the difference column so the most low-risk-distinctive terms come first.
print("Top 25 terms more common in LOW-RISK reviews:")
print(comparison.sort_values('difference', ascending=True).head(25).to_string(index=False))

Top 25 terms more common in LOW-RISK reviews:
          term  high_risk_pct  low_risk_pct  difference
         great            2.8          26.1       -23.4
       helpful            2.6          25.5       -22.9
          good            8.2          24.2       -16.0
         happy            2.0          13.7       -11.7
      friendly            0.5          11.8       -11.3
         thank            1.1           9.8        -8.7
          easy            1.4           9.8        -8.4
       patient            0.3           8.5        -8.2
         store            7.4          15.0        -7.7
       amazing            0.3           7.2        -6.9
 great service            0.0           6.5        -6.5
     excellent            0.8           7.2        -6.4
     fantastic            0.2           5.9        -5.7
  professional            0.5           5.9        -5.4
          team            5.7          11.1        -5.4
         today            4.7           9.8        -5.1
  

The high-risk terms are then ranked by TF-IDF weight within the high-risk
reviews, which weighs each term by frequency and distinctiveness together. This
complements the proportion analysis: a term ranking highly on both measures is a
robust driver. The ranked list provides the ordered set of service-related
drivers that the study set out to identify.

In [77]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Compute TF-IDF within the high-risk reviews only, to rank the terms that carry
# most weight in high-risk feedback.
high_risk_texts = labelled[labelled['churn_risk'] == 1]['text_topics']

tfidf_hr = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.5
)
hr_matrix = tfidf_hr.fit_transform(high_risk_texts)
hr_terms = tfidf_hr.get_feature_names_out()

# Sum each term's TF-IDF weight across all high-risk reviews, then rank.
hr_scores = np.asarray(hr_matrix.sum(axis=0)).ravel()
hr_ranking = pd.DataFrame({
    'term': hr_terms,
    'tfidf_score': hr_scores.round(2)
}).sort_values('tfidf_score', ascending=False).reset_index(drop=True)

print("Top 25 high-risk drivers by TF-IDF weight:")
print(hr_ranking.head(25).to_string(index=False))

Top 25 high-risk drivers by TF-IDF weight:
            term  tfidf_score
        customer        77.10
           phone        58.82
customer service        56.17
             bad        51.97
            time        51.44
            plan        47.94
           month        40.32
        internet        37.65
         account        37.63
          charge        37.03
             day        36.54
        provider        36.27
           issue        33.91
            year        32.64
            hour        31.95
        terrible        28.62
            bill        28.52
           money        27.39
          number        27.37
          cancel        27.20
           datum        26.05
            week        25.77
       broadband        25.45
           email        24.97
          mobile        23.69


## 5. Topic Modelling with LDA

Latent Dirichlet Allocation (LDA) identifies themes in a collection of documents
by finding groups of words that tend to occur together, without the themes being
specified in advance. Each theme is a distribution of words, and each document is
treated as a mixture of themes. Applied to the feedback, LDA surfaces the themes
that customers raise, which addresses the question of what themes are present in
the feedback.

Following the difference in purpose between the two platforms, LDA is applied
separately to each source. The Trustpilot model is the main analysis, given its
larger corpus, while the Geekzone model is complementary and more exploratory,
given its smaller corpus. The number of themes for each model is chosen by
examining topic coherence across a range of values together with the
interpretability of the resulting themes.

In [78]:
# Install gensim for LDA topic modelling. It provides the LDA model and the
# coherence measure used to help choose the number of topics.
!pip install gensim --break-system-packages --quiet

### 5.1 Trustpilot Topic Model

The Trustpilot reviews in the topic corpus are prepared for LDA. The processed
text of each review is split into a list of words, and two structures required
by gensim are built from these: a dictionary, which maps every word to an
identifier, and a corpus in bag-of-words form, which represents each review as
the set of word identifiers it contains together with their counts. The number
of topics is then chosen by examining coherence across a range of values.

In [79]:
from gensim import corpora

# Take the Trustpilot reviews from the topic corpus (>= 10 content words).
tp_topic = topic_corpus[topic_corpus['source'] == 'Trustpilot'].copy()

# Split each processed text into a list of words. gensim works with lists of
# tokens, not strings.
tp_tokens = tp_topic['text_topics'].apply(lambda t: t.split()).tolist()

# Build the dictionary: it assigns an integer id to every distinct word.
tp_dictionary = corpora.Dictionary(tp_tokens)

# Filter extremes to clean the vocabulary:
#   no_below=5   -> drop words appearing in fewer than 5 reviews (rare noise)
#   no_above=0.5 -> drop words appearing in more than 50% of reviews (too common)
tp_dictionary.filter_extremes(no_below=5, no_above=0.5)

# Build the bag-of-words corpus: each review becomes a list of (word_id, count).
tp_corpus = [tp_dictionary.doc2bow(tokens) for tokens in tp_tokens]

print(f"Trustpilot reviews for LDA: {len(tp_tokens)}")
print(f"Vocabulary size after filtering: {len(tp_dictionary)}")
print()
# Show what a bag-of-words review looks like, for understanding.
print("Example: first review as (word_id, count) pairs:")
print(tp_corpus[0][:10])
print()
print("The same ids translated back to words:")
print([(tp_dictionary[wid], count) for wid, count in tp_corpus[0][:10]])

Trustpilot reviews for LDA: 1103
Vocabulary size after filtering: 746

Example: first review as (word_id, count) pairs:
[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1)]

The same ids translated back to words:
[('control', 1), ('day', 1), ('door', 1), ('hold', 1), ('minute', 1), ('open', 1), ('poor', 1), ('shop', 1)]


In [80]:
from gensim.models import LdaModel, CoherenceModel

# Test a range of topic numbers and measure coherence for each, to inform the
# choice of k. For each k we train an LDA model and compute its coherence.
# random_state fixes the randomness so results are reproducible.
k_range = range(3, 11)   # test 3 to 10 topics
coherence_scores = []

for k in k_range:
    model = LdaModel(
        corpus=tp_corpus,
        id2word=tp_dictionary,
        num_topics=k,
        random_state=42,
        passes=10,          # how many times it goes over the corpus while learning
        alpha='auto'        # let the model learn how documents mix topics
    )
    # 'c_v' is a widely used coherence measure; higher is better.
    coherence = CoherenceModel(
        model=model,
        texts=tp_tokens,
        dictionary=tp_dictionary,
        coherence='c_v'
    ).get_coherence()
    coherence_scores.append(coherence)
    print(f"k={k}: coherence = {coherence:.4f}")

print()
best_k = k_range[coherence_scores.index(max(coherence_scores))]
print(f"Highest coherence at k={best_k}")

k=3: coherence = 0.3717
k=4: coherence = 0.3478
k=5: coherence = 0.3351
k=6: coherence = 0.3273
k=7: coherence = 0.3222
k=8: coherence = 0.3313
k=9: coherence = 0.3310
k=10: coherence = 0.3086

Highest coherence at k=3


In [81]:
def show_topics(k, corpus, dictionary, tokens):
    """Train an LDA with k topics and print the top words of each topic."""
    model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=10,
        alpha='auto'
    )
    print(f"===== LDA with k={k} topics =====")
    for topic_id in range(k):
        # Get the top 10 words for this topic, by weight.
        words = model.show_topic(topic_id, topn=10)
        word_list = ', '.join([w for w, weight in words])
        print(f"Topic {topic_id + 1}: {word_list}")
    print()
    return model

# Look at the two candidates side by side.
model_k3 = show_topics(3, tp_corpus, tp_dictionary, tp_tokens)
model_k8 = show_topics(8, tp_corpus, tp_dictionary, tp_tokens)

===== LDA with k=3 topics =====
Topic 1: time, phone, hour, day, week, internet, datum, account, minute, call
Topic 2: bad, account, month, issue, time, money, year, contract, internet, broadband
Topic 3: phone, plan, time, charge, month, number, bill, issue, provider, day

===== LDA with k=8 topics =====
Topic 1: time, phone, datum, bad, plan, hour, credit, day, text, app
Topic 2: month, broadband, mobile, provider, contract, bill, modem, issue, bad, account
Topic 3: phone, plan, bill, month, time, charge, provider, bad, email, payment
Topic 4: phone, plan, day, cancel, contract, month, charge, mobile, email, account
Topic 5: month, day, phone, issue, plan, time, internet, mobile, charge, number
Topic 6: internet, issue, modem, connection, account, time, bad, experience, fibre, phone
Topic 7: time, account, phone, hour, week, number, charge, year, person, day
Topic 8: phone, store, time, year, call, business, day, account, bad, money



In [82]:
# LDA-specific stopwords: domain terms that appear across almost all reviews and
# blur the topics without naming a theme. This list is used only for LDA, not
# for TF-IDF. Starting conservatively with the four clearest cases.
lda_extra_stopwords = {'phone', 'time', 'day', 'issue'}

# Rebuild the Trustpilot token lists, removing the LDA stopwords. We start from
# the already-processed text_topics, so this only drops the extra terms.
tp_tokens_lda = tp_topic['text_topics'].apply(
    lambda t: [w for w in t.split() if w not in lda_extra_stopwords]
).tolist()

# Rebuild the dictionary and corpus with the reduced tokens.
tp_dictionary = corpora.Dictionary(tp_tokens_lda)
tp_dictionary.filter_extremes(no_below=5, no_above=0.5)
tp_corpus = [tp_dictionary.doc2bow(tokens) for tokens in tp_tokens_lda]

print(f"Trustpilot reviews for LDA: {len(tp_tokens_lda)}")
print(f"Vocabulary size after removing LDA stopwords and filtering: {len(tp_dictionary)}")

Trustpilot reviews for LDA: 1103
Vocabulary size after removing LDA stopwords and filtering: 742


In [83]:
# Re-examine the topics after removing the four vague terms.
model_k3 = show_topics(3, tp_corpus, tp_dictionary, tp_tokens_lda)
model_k8 = show_topics(8, tp_corpus, tp_dictionary, tp_tokens_lda)

===== LDA with k=3 topics =====
Topic 1: plan, account, month, charge, bill, credit, year, money, datum, email
Topic 2: internet, store, bad, week, number, month, broadband, provider, hour, staff
Topic 3: provider, charge, month, good, datum, problem, account, sim, bad, year

===== LDA with k=8 topics =====
Topic 1: plan, bill, account, month, money, credit, charge, payment, year, email
Topic 2: week, store, month, modem, charge, people, provider, hour, email, number
Topic 3: account, charge, month, bill, year, good, provider, people, email, money
Topic 4: plan, number, broadband, mobile, staff, store, experience, connection, hour, support
Topic 5: internet, speed, connection, provider, fibre, plan, router, week, hour, slow
Topic 6: datum, app, charge, credit, card, staff, month, mobile, provider, plan
Topic 7: charge, cancel, month, account, hour, bill, person, problem, leave, mobile
Topic 8: bad, plan, year, contract, month, provider, account, problem, mobile, email



In [84]:
# Test the middle range, where the real themes may separate cleanly without the
# redundancy seen at k=8.
model_k5 = show_topics(5, tp_corpus, tp_dictionary, tp_tokens_lda)
model_k6 = show_topics(6, tp_corpus, tp_dictionary, tp_tokens_lda)

===== LDA with k=5 topics =====
Topic 1: account, plan, month, charge, bill, credit, money, year, email, payment
Topic 2: month, bad, store, week, contract, charge, people, modem, email, provider
Topic 3: charge, month, bill, provider, year, problem, sim, datum, account, good
Topic 4: plan, mobile, number, broadband, staff, datum, bad, app, store, provider
Topic 5: internet, speed, connection, plan, hour, provider, fibre, week, bad, address

===== LDA with k=6 topics =====
Topic 1: plan, account, month, year, money, bill, credit, charge, email, payment
Topic 2: month, store, week, charge, bad, cancel, email, contract, people, modem
Topic 3: month, account, provider, bill, year, charge, problem, payment, sim, good
Topic 4: plan, number, mobile, broadband, staff, bad, experience, hour, provider, store
Topic 5: internet, speed, connection, plan, hour, fibre, provider, bad, week, month
Topic 6: datum, charge, month, bill, plan, account, credit, app, text, mobile



In [85]:
# Test with a lower content-word threshold, to see whether more documents
# changes the topic overlap problem.
tp_topic_lower = df[(df['source'] == 'Trustpilot') & (df['content_word_count'] >= 7)].copy()

tp_tokens_lower = tp_topic_lower['text_topics'].apply(
    lambda t: [w for w in t.split() if w not in lda_extra_stopwords]
).tolist()

tp_dict_lower = corpora.Dictionary(tp_tokens_lower)
tp_dict_lower.filter_extremes(no_below=5, no_above=0.5)
tp_corpus_lower = [tp_dict_lower.doc2bow(tokens) for tokens in tp_tokens_lower]

print(f"Trustpilot reviews with threshold 7: {len(tp_tokens_lower)}")
print()

model_k5_lower = show_topics(5, tp_corpus_lower, tp_dict_lower, tp_tokens_lower)

Trustpilot reviews with threshold 7: 1304

===== LDA with k=5 topics =====
Topic 1: plan, customer, datum, number, year, account, speed, app, charge, card
Topic 2: plan, customer, bill, account, bad, charge, cancel, month, datum, provider
Topic 3: customer, store, week, number, staff, people, provider, bad, hour, month
Topic 4: month, charge, modem, contract, account, broadband, email, year, plan, cancel
Topic 5: customer, internet, bad, support, hour, provider, experience, terrible, connection, hold



In [86]:
# Fix the final Trustpilot LDA model: threshold 10, k=5, LDA-specific stopwords
# removed. Rebuild from the threshold-10 corpus to make sure the final model
# uses the confirmed settings.
tp_topic = topic_corpus[topic_corpus['source'] == 'Trustpilot'].copy()

tp_tokens_lda = tp_topic['text_topics'].apply(
    lambda t: [w for w in t.split() if w not in lda_extra_stopwords]
).tolist()

tp_dictionary = corpora.Dictionary(tp_tokens_lda)
tp_dictionary.filter_extremes(no_below=5, no_above=0.5)
tp_corpus = [tp_dictionary.doc2bow(tokens) for tokens in tp_tokens_lda]

# Train the final model with k=5.
tp_lda_model = LdaModel(
    corpus=tp_corpus,
    id2word=tp_dictionary,
    num_topics=5,
    random_state=42,
    passes=10,
    alpha='auto'
)

print("Final Trustpilot LDA model fixed: 5 topics.")
print()
for topic_id in range(5):
    words = tp_lda_model.show_topic(topic_id, topn=10)
    word_list = ', '.join([w for w, weight in words])
    print(f"Topic {topic_id + 1}: {word_list}")

Final Trustpilot LDA model fixed: 5 topics.

Topic 1: account, plan, month, charge, bill, credit, money, year, email, payment
Topic 2: month, bad, store, week, contract, charge, people, modem, email, provider
Topic 3: charge, month, bill, provider, year, problem, sim, datum, account, good
Topic 4: plan, mobile, number, broadband, staff, datum, bad, app, store, provider
Topic 5: internet, speed, connection, plan, hour, provider, fibre, week, bad, address


In [87]:
# For each review, find its dominant topic: the topic with the highest
# probability in the document's topic mixture.
topic_assignments = []
for bow in tp_corpus:
    topic_probs = tp_lda_model.get_document_topics(bow)
    # topic_probs is a list of (topic_id, probability). Pick the highest.
    dominant_topic = max(topic_probs, key=lambda x: x[1])[0]
    topic_assignments.append(dominant_topic)

tp_topic['dominant_topic'] = topic_assignments

# Name the topics based on the interpretation above.
topic_names = {
    0: 'Billing and Payments',
    1: 'Contracts and In-Store Experience',
    2: 'General Service Evaluation',
    3: 'Mobile Plans and Data Usage',
    4: 'Network Speed and Connectivity'
}
tp_topic['topic_name'] = tp_topic['dominant_topic'].map(topic_names)

# How many reviews fall into each topic, and their average VADER sentiment.
topic_summary = tp_topic.groupby('topic_name').agg(
    review_count=('text', 'count'),
    avg_compound=('vader_compound', 'mean'),
    pct_high_risk=('rating', lambda r: (r <= 2).mean() * 100)
).round(3).sort_values('review_count', ascending=False)

print("Topic summary: review count, average sentiment, and % high-risk (1-2 stars):")
print(topic_summary)

Topic summary: review count, average sentiment, and % high-risk (1-2 stars):
                                   review_count  avg_compound  pct_high_risk
topic_name                                                                  
Contracts and In-Store Experience           286        -0.329         91.958
Billing and Payments                        276        -0.323         94.565
Mobile Plans and Data Usage                 219        -0.088         86.758
Network Speed and Connectivity              178        -0.337         96.067
General Service Evaluation                  144        -0.106         81.944


In [89]:
# Check for duplicate column names, the likely cause of the error.
print("Columns in tp_topic:")
print(list(tp_topic.columns))
print()
print("Duplicated column names:")
print(tp_topic.columns[tp_topic.columns.duplicated()].tolist())
print()
print("Shape:", tp_topic.shape)

Columns in tp_topic:
['provider', 'source', 'rating', 'title', 'text', 'date', 'url', 'text_clean', 'word_count', 'vader_neg', 'vader_neu', 'vader_pos', 'vader_compound', 'text_topics_v1', 'content_word_count', 'text_topics', 'dominant_topic', 'topic_name', 'is_high_risk']

Duplicated column names:
[]

Shape: (1103, 19)


### 5.2 Geekzone Topic Model

The Geekzone posts in the topic corpus are modelled separately, as a
complementary and more exploratory analysis, given the smaller corpus and the
platform's technical discussion focus. The same preparation is used: the
processed text is tokenised, a dictionary and bag-of-words corpus are built, and
the number of topics is chosen from coherence across a range of values together
with interpretability. As Geekzone posts carry no star rating, the topics are
not cross-tabulated with churn risk; instead they describe the themes that arise
in technical community discussion of the three providers.

In [90]:
# Prepare the Geekzone posts from the topic corpus.
gz_topic = topic_corpus[topic_corpus['source'] == 'Geekzone'].copy()

# Apply the same LDA-specific stopwords used for Trustpilot, for consistency.
gz_tokens_lda = gz_topic['text_topics'].apply(
    lambda t: [w for w in t.split() if w not in lda_extra_stopwords]
).tolist()

# Build dictionary and corpus.
gz_dictionary = corpora.Dictionary(gz_tokens_lda)
gz_dictionary.filter_extremes(no_below=5, no_above=0.5)
gz_corpus = [gz_dictionary.doc2bow(tokens) for tokens in gz_tokens_lda]

print(f"Geekzone posts for LDA: {len(gz_tokens_lda)}")
print(f"Vocabulary size after filtering: {len(gz_dictionary)}")
print()

# Measure coherence across a range of k, as we did for Trustpilot.
print("Coherence by number of topics (Geekzone):")
gz_coherence = []
for k in range(3, 9):
    model = LdaModel(
        corpus=gz_corpus,
        id2word=gz_dictionary,
        num_topics=k,
        random_state=42,
        passes=10,
        alpha='auto'
    )
    coh = CoherenceModel(
        model=model,
        texts=gz_tokens_lda,
        dictionary=gz_dictionary,
        coherence='c_v'
    ).get_coherence()
    gz_coherence.append(coh)
    print(f"  k={k}: coherence = {coh:.4f}")

Geekzone posts for LDA: 276
Vocabulary size after filtering: 305

Coherence by number of topics (Geekzone):
  k=3: coherence = 0.3211
  k=4: coherence = 0.3622
  k=5: coherence = 0.3259
  k=6: coherence = 0.3512
  k=7: coherence = 0.3286
  k=8: coherence = 0.3374


In [91]:
# Look at the two coherence peaks for Geekzone.
print("GEEKZONE TOPICS")
print()
gz_model_k4 = show_topics(4, gz_corpus, gz_dictionary, gz_tokens_lda)
gz_model_k6 = show_topics(6, gz_corpus, gz_dictionary, gz_tokens_lda)

GEEKZONE TOPICS

===== LDA with k=4 topics =====
Topic 1: router, g, switch, device, speed, isp, case, network, video, ethernet
Topic 2: number, month, plan, error, power, good, connection, mobile, thing, office
Topic 3: plan, device, app, router, g, network, customer, service, account, problem
Topic 4: modem, connection, router, network, fibre, setting, cable, wireless, internet, address

===== LDA with k=6 topics =====
Topic 1: g, router, switch, problem, call, charge, device, people, interface, location
Topic 2: plan, number, month, mobile, customer, work, worth, leave, cheap, monthly
Topic 3: device, network, app, error, router, message, g, datum, connection, account
Topic 4: connection, modem, fibre, setting, wireless, router, network, cable, home, internet
Topic 5: router, port, office, sure, option, credit, firmware, thank, user, point
Topic 6: router, power, video, dns, service, speed, network, connection, hour, good



In [92]:
# Fix the final Geekzone LDA model with k=4.
gz_lda_model = LdaModel(
    corpus=gz_corpus,
    id2word=gz_dictionary,
    num_topics=4,
    random_state=42,
    passes=10,
    alpha='auto'
)

# Assign each post to its dominant topic.
gz_topic_assignments = []
for bow in gz_corpus:
    topic_probs = gz_lda_model.get_document_topics(bow)
    dominant_topic = max(topic_probs, key=lambda x: x[1])[0]
    gz_topic_assignments.append(dominant_topic)

gz_topic = gz_topic.loc[:, ~gz_topic.columns.duplicated()].copy()
gz_topic['dominant_topic'] = gz_topic_assignments

gz_topic_names = {
    0: 'Home Network Setup and Equipment',
    1: 'Mobile Plans and Account Issues',
    2: 'App and Customer Service Problems',
    3: 'Fibre and Connection Configuration'
}
gz_topic['topic_name'] = gz_topic['dominant_topic'].map(gz_topic_names)

# Summarise: post count and average VADER sentiment per topic (no risk label
# available for Geekzone).
gz_summary = gz_topic.groupby('topic_name').agg(
    post_count=('text', 'count'),
    avg_compound=('vader_compound', 'mean')
).round(3).sort_values('post_count', ascending=False)

print("Geekzone topic summary:")
print(gz_summary)

Geekzone topic summary:
                                    post_count  avg_compound
topic_name                                                  
Fibre and Connection Configuration          88         0.326
Mobile Plans and Account Issues             70         0.535
App and Customer Service Problems           62         0.380
Home Network Setup and Equipment            56         0.408


## 6. Saving Processed Data for Modelling

The columns produced in this notebook, VADER sentiment scores, the churn-risk
proxy, the processed text for TF-IDF and LDA, and the Trustpilot topic
assignment, are stored back in the database in a new table, `reviews_processed`.
This keeps the raw consolidated data in `reviews` unchanged and separates it
from the analysis output, while allowing the modelling notebook to read the
processed data directly rather than repeating the sentiment and pre-processing
steps.

In [94]:
import sqlite3

# Build the churn_risk proxy on the full df, matching what we defined earlier
# for the Trustpilot subset. Rows outside Trustpilot, or 3-star reviews, get a
# missing value, consistent with the proxy definition.
def assign_risk(row):
    if row['source'] != 'Trustpilot':
        return None
    if row['rating'] in (1, 2):
        return 1
    elif row['rating'] in (4, 5):
        return 0
    else:
        return None

df['churn_risk'] = df.apply(assign_risk, axis=1)

# Bring in the Trustpilot dominant topic from tp_topic, matched by row. tp_topic
# was built from topic_corpus, a subset of df, so we merge on the index or on a
# unique identifier. Here we use the DataFrame index, assuming it was preserved.
df['dominant_topic_trustpilot'] = None
df.loc[tp_topic.index, 'dominant_topic_trustpilot'] = tp_topic['topic_name']

# Also bring in the Geekzone dominant topic, the same way.
df['dominant_topic_geekzone'] = None
df.loc[gz_topic.index, 'dominant_topic_geekzone'] = gz_topic['topic_name']

# Save the processed dataset to a new table, replacing it if this cell is
# re-run.
connection = sqlite3.connect('telecom_reviews.db')
df.to_sql('reviews_processed', con=connection, if_exists='replace', index=False)
connection.commit()
connection.close()

print("Saved 'reviews_processed' table.")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
print("Non-null churn_risk (Trustpilot, 1-2 or 4-5 stars):", df['churn_risk'].notna().sum())
print("Non-null dominant_topic_trustpilot:", df['dominant_topic_trustpilot'].notna().sum())
print("Non-null dominant_topic_geekzone:", df['dominant_topic_geekzone'].notna().sum())

Saved 'reviews_processed' table.
Rows: 1929
Columns: ['provider', 'source', 'rating', 'title', 'text', 'date', 'url', 'text_clean', 'word_count', 'vader_neg', 'vader_neu', 'vader_pos', 'vader_compound', 'text_topics_v1', 'content_word_count', 'text_topics', 'churn_risk', 'dominant_topic_trustpilot', 'dominant_topic_geekzone']

Non-null churn_risk (Trustpilot, 1-2 or 4-5 stars): 1483
Non-null dominant_topic_trustpilot: 1103
Non-null dominant_topic_geekzone: 276
